In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.nn.init as init
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import trange
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

### Load the Credit Data

In [ ]:
filename = 'default of credit card clients.csv'
credit_data = pd.read_csv(filename)

In [ ]:
credit_data.head()

### Prepare the Dataset

In [ ]:
credit_data.drop(['ID'], axis=1, inplace=True)

In [ ]:
# Identify number of categories in all categorical variables
categorical_list = ['SEX', 'EDUCATION', 'MARRIAGE']
credit_data[categorical_list].nunique()

In [ ]:
# One-hot encode
credit_cat_cols = pd.get_dummies(credit_data, columns = categorical_list)

In [ ]:
credit_cat_cols.head()

In [ ]:
credit_cat_cols.columns

In [ ]:
y = credit_cat_cols['default payment next month'].to_numpy()
X = credit_cat_cols.drop(['default payment next month'], axis=1)

### Test-Train-Validation Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Standard Scaling

In [ ]:
standard_scaler = StandardScaler()
scaler_cols = list(set(credit_data.columns) - set(categorical_list))
scaler_cols.remove('default payment next month')
X_train_scaled = standard_scaler.fit_transform(X_train[scaler_cols])
X_train[scaler_cols] = X_train_scaled

In [ ]:
X_test_scaled = standard_scaler.transform(X_test[scaler_cols])
X_test[scaler_cols] = X_test_scaled

In [ ]:
input_cols = X_train.shape[1]
input_cols

In [ ]:
no_epochs = 100
batch_size = 100

In [ ]:
def createDataLoader(X, y, shuffle):
    tx = torch.Tensor(X.to_numpy().astype(np.float32).copy()).to(device)
    ty = torch.Tensor(y.to_numpy().astype(np.float32).copy()).to(device)
    dataset = TensorDataset(tx, ty)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, 
                                         shuffle=shuffle, drop_last=True)
    return loader

### Build Model

In [ ]:
def train_model(model, train_loader, test_loader, loss_fn, optimizer, epochs, level = 0.5):
    train_errors = []
    test_errors = []
    train_accuracies = []
    test_accuracies = []

    tqdm_epoch = trange(epochs)
    for epoch in tqdm_epoch:
        model.train()
        train_loss = 0.0
        correct_train = 0

        # Training
        for batch_X, batch_y in train_loader:
            # Forward pass
            outputs = model(batch_X)
            loss = loss_fn(outputs.squeeze(), batch_y.float())

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * batch_X.size(0)

            # Calculate training accuracy
            predicted = outputs.squeeze() > level
            correct_train += (predicted == batch_y).sum().item()

        train_loss /= len(train_loader.dataset)
        train_accuracy = 100 * correct_train / len(train_loader.dataset)
        train_errors.append(train_loss)
        train_accuracies.append(train_accuracy)

        # Evaluation on test set
        model.eval()
        test_loss = 0.0
        correct_test = 0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                outputs = model(batch_X)
                loss = loss_fn(outputs.squeeze(), batch_y.float())
                test_loss += loss.item() * batch_X.size(0)

                # Calculate test accuracy
                predicted = outputs.squeeze() > level
                correct_test += (predicted == batch_y).sum().item()

        test_loss /= len(test_loader.dataset)
        test_accuracy = 100 * correct_test / len(test_loader.dataset)
        test_errors.append(test_loss)
        test_accuracies.append(test_accuracy)

        tqdm_epoch.set_description(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}, Train Acc: {train_accuracy:.2f}%, Test Acc: {test_accuracy:.2f}%")

    history = {}
    history['train_loss'] = train_errors
    history['test_loss'] = test_errors
    history['train_acc'] = train_accuracies
    history['test_acc'] = test_accuracies
        
    return history

In [ ]:
class BinaryClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, output_activation='sigmoid'):
        super(BinaryClassifier, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, 1)

        if output_activation == 'sigmoid':
            self.output_activation = nn.Sigmoid()
        elif output_activation == 'tanh':
            self.output_activation = nn.Tanh()
        else:
            raise ValueError("Invalid output activation. Choose 'sigmoid' or 'tanh'.")

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.output_layer(x)
        x = self.output_activation(x)
        return x


### K-Fold Cross-Validation

In [ ]:
from sklearn.model_selection import KFold

In [ ]:
no_splits = 5
kf = KFold(n_splits=no_splits)

In [ ]:
X_train = X_train.to_numpy()

In [ ]:
all_histories = []
current_split = 0
for train_index, val_index in kf.split(X_train, y_train):
    iX_train, iX_val = X_train[train_index], X_train[val_index]
    iy_train, iy_val = y_train[train_index], y_train[val_index]
    train_loader = createDataLoader(iX_train, iy_train, True)
    val_loader = createDataLoader(iX_val, iy_val, False)
    
    print("Processing fold: {0}".format(current_split))
    current_split = current_split + 1
    
    model = BinaryClassifier(input_cols, 50).to(device)
    loss_fn = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    history_dict = train_model(model, train_loader, val_loader,
                       loss_fn, optimizer, no_epochs)
    val_history = history_dict['test_acc']
    all_histories.append(val_history)

In [ ]:
average_history = [np.mean([x[i] for x in all_histories]) for i in range(no_epochs)]

### Validation Accuracy

In [ ]:
epochs = range(1, no_epochs + 1)

In [ ]:
plt.figure(figsize=(10,7)) 
    
plt.plot(epochs, average_history, color='k', linestyle='-', label = 'K-Fold')

plt.xlabel('Epochs')
plt.ylabel('Validation Accuracy')
plt.ylim(78, 83)
plt.legend()
plt.savefig('TestTCKFoldValidAcc.png', dpi=300, bbox_inches='tight')

### Retrain using whole training set

In [ ]:
train_loader = createDataLoader(X_train, y_train, True)
val_loader = createDataLoader(X_test.to_numpy(), y_test, False)
    
model = BinaryClassifier(input_cols, 50).to(device)
loss_fn = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
history = train_model(model, train_loader, val_loader,
                       loss_fn, optimizer, no_epochs)

### Evaluation Metrics

In [ ]:
tx = torch.Tensor(X_test.to_numpy()).to(device)
y_predict = model(tx)
y_pred = np.array([1 if i > 0.5 else 0 for i in y_predict])

In [ ]:
# Accuracy
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy = {}".format(accuracy))

In [ ]:
# Balanced Accuracy
from sklearn.metrics import balanced_accuracy_score
balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
print("Balanced Accuracy = {}".format(balanced_accuracy))

In [ ]:
# Precision
from sklearn.metrics import precision_score
precision = precision_score(y_test, y_pred)
print("Precision = {}".format(precision))

In [ ]:
# Recall
from sklearn.metrics import recall_score
recall = recall_score(y_test, y_pred)
print("Recall = {}".format(recall))

In [ ]:
# F1-Score
from sklearn.metrics import f1_score
f1 = f1_score(y_test, y_pred)
print("F1-Score = {}".format(f1))

In [ ]:
# AUC
from sklearn.metrics import roc_auc_score
# Note use of raw probabilities
auc = roc_auc_score(y_test, y_pred)
print("AUC = {}".format(auc))

### ROC curve

In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(y_test, y_pred)

In [ ]:
plt.figure(figsize=(7,5))
plt.plot(fpr, tpr, color='k', label='ROC curve')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc="lower right")
plt.savefig('TestTCROCCurve.png', dpi=300, bbox_inches='tight')